# Phase 8: Hybrid Recommendation System

## Objective
Develop a hybrid movie recommendation system by combining
Content-Based Filtering and Collaborative Filtering.

## Approach
- Generate personalized content-based scores using a user's
  previously liked movies.
- Generate collaborative scores using similar users.
- Normalize both scores to a common scale.
- Combine the scores using a weighted formula.
- Rank movies and recommend the top results.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from scipy.sparse import csr_matrix, lil_matrix
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data" / "processed"
FEATURE_DIR = DATA_DIR / "features"

print("Project directory:", BASE_DIR)
print("Processed data directory:", DATA_DIR)

Project directory: /home/shahzad-mustafa/Desktop/MovieRecommendationSystem
Processed data directory: /home/shahzad-mustafa/Desktop/MovieRecommendationSystem/data/processed


In [3]:
movies = pd.read_csv(DATA_DIR / "movies_clean.csv")
ratings = pd.read_csv(DATA_DIR / "ratings_clean.csv")

genre_features = pd.read_csv(
    FEATURE_DIR / "movie_genre_features.csv",
    index_col="MovieID"
)

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Genre features:", genre_features.shape)

Movies: (3883, 3)
Ratings: (1000209, 5)
Genre features: (3883, 18)


In [4]:
movies["MovieID"] = movies["MovieID"].astype(int)
ratings["MovieID"] = ratings["MovieID"].astype(int)
ratings["UserID"] = ratings["UserID"].astype(int)

genre_features.index = genre_features.index.astype(int)

# Keep only movies that exist in both datasets
common_movie_ids = movies[
    movies["MovieID"].isin(genre_features.index)
]["MovieID"]

genre_features = genre_features.loc[common_movie_ids].sort_index()

movies = movies.set_index("MovieID").loc[genre_features.index]

print("Movies:", movies.shape)
print("Genre features:", genre_features.shape)
print("Movie IDs aligned:", movies.index.equals(genre_features.index))

Movies: (3883, 2)
Genre features: (3883, 18)
Movie IDs aligned: True


In [5]:
user_id = 1

user_ratings = ratings[ratings["UserID"] == user_id].copy()

print("Selected User ID:", user_id)
print("Movies rated:", len(user_ratings))
print("\nRating distribution:")
print(user_ratings["Rating"].value_counts().sort_index())

Selected User ID: 1
Movies rated: 53

Rating distribution:
Rating
3.0     8
4.0    27
5.0    18
Name: count, dtype: int64


In [6]:
liked_movies = user_ratings[
    user_ratings["Rating"] >= 4
]["MovieID"]

liked_movie_features = genre_features.loc[
    genre_features.index.intersection(liked_movies)
]

user_profile = liked_movie_features.mean(axis=0)

print("Liked movies:", len(liked_movies))
print("Profile shape:", user_profile.shape)
print("\nUser genre preferences:")
print(user_profile.sort_values(ascending=False))

Liked movies: 45
Profile shape: (18,)

User genre preferences:
Drama          0.466667
Children's     0.400000
Animation      0.311111
Musical        0.266667
Comedy         0.244444
Action         0.088889
Adventure      0.088889
Fantasy        0.066667
Romance        0.066667
Sci-Fi         0.066667
War            0.044444
Thriller       0.044444
Crime          0.044444
Documentary    0.000000
Film-Noir      0.000000
Horror         0.000000
Mystery        0.000000
Western        0.000000
dtype: float64


In [8]:
# Convert the user's genre profile into a 2D array
user_profile_array = user_profile.values.reshape(1, -1)

# Calculate cosine similarity between user profile and all movies
content_similarity_scores = cosine_similarity(
    user_profile_array,
    genre_features
)[0]

# Store scores with their MovieIDs
content_scores = pd.Series(
    content_similarity_scores,
    index=genre_features.index
)

# Exclude movies already rated by the user
content_scores = content_scores[
    ~content_scores.index.isin(watched_movie_ids)
]

# Sort recommendations
content_scores = content_scores.sort_values(ascending=False)

content_recommendations = pd.DataFrame({
    "MovieID": content_scores.index,
    "ContentScore": content_scores.values
})

content_recommendations = content_recommendations.merge(
    movies.reset_index()[["MovieID", "Title", "Genres"]],
    on="MovieID",
    how="left"
)

content_recommendations.head(10)

,MovieID,ContentScore,Title,Genres
0,1812,0.801566,Wide Awake (1998),Children's|Comedy|Drama
1,34,0.801566,Babe (1995),Children's|Comedy|Drama
2,1014,0.801566,Pollyanna (1960),Children's|Comedy|Drama
3,2138,0.777478,Watership Down (1978),Animation|Children's|Drama|Fantasy
4,262,0.765737,"Little Princess, A (1995)",Children's|Drama
5,2079,0.765737,Kidnapped (1960),Children's|Drama
6,755,0.765737,Kim (1950),Children's|Drama
7,1013,0.765737,"Parent Trap, The (1961)",Children's|Drama
8,1012,0.765737,Old Yeller (1957),Children's|Drama
9,2400,0.765737,Prancer (1989),Children's|Drama


In [9]:
user_movie_matrix = ratings.pivot(
    index="UserID",
    columns="MovieID",
    values="Rating"
)

print("User-movie matrix shape:", user_movie_matrix.shape)
print("Missing values:", user_movie_matrix.isna().sum().sum())

User-movie matrix shape: (6040, 3706)
Missing values: 21384031


In [10]:
user_means = user_movie_matrix.mean(axis=1)

centered_matrix = user_movie_matrix.sub(
    user_means,
    axis=0
).fillna(0)

print("User average ratings:")
print(user_means.head())

print("\nCentered matrix shape:", centered_matrix.shape)

User average ratings:
UserID
1    4.188679
2    3.713178
3    3.901961
4    4.190476
5    3.146465
dtype: float64

Centered matrix shape: (6040, 3706)


In [11]:
centered_sparse = csr_matrix(
    centered_matrix.to_numpy(dtype=np.float32)
)

rating_presence = csr_matrix(
    user_movie_matrix.notna().to_numpy(dtype=np.float32)
)

print("Centered sparse matrix:", centered_sparse.shape)
print("Stored rating values:", centered_sparse.nnz)

Centered sparse matrix: (6040, 3706)
Stored rating values: 998088


In [12]:
similarity_matrix = cosine_similarity(
    centered_sparse,
    dense_output=False
).tocsr()

top20_similarity = lil_matrix(
    similarity_matrix.shape,
    dtype=np.float32
)

for user_index in range(similarity_matrix.shape[0]):

    similarity_row = similarity_matrix.getrow(user_index)

    neighbor_indices = similarity_row.indices
    neighbor_scores = similarity_row.data

    # Exclude the user themselves and non-positive similarities
    valid_mask = (
        (neighbor_indices != user_index) &
        (neighbor_scores > 0)
    )

    neighbor_indices = neighbor_indices[valid_mask]
    neighbor_scores = neighbor_scores[valid_mask]

    # Keep the 20 most similar users
    if len(neighbor_scores) > 20:
        top_indices = np.argsort(neighbor_scores)[-20:]

        neighbor_indices = neighbor_indices[top_indices]
        neighbor_scores = neighbor_scores[top_indices]

    top20_similarity[user_index, neighbor_indices] = neighbor_scores

top20_similarity = top20_similarity.tocsr()

print("Similarity matrix shape:", top20_similarity.shape)
print("Non-zero neighbor similarities:", top20_similarity.nnz)

Similarity matrix shape: (6040, 6040)
Non-zero neighbor similarities: 120800


In [13]:
weighted_deviations_matrix = (
    top20_similarity.dot(centered_sparse)
).tocsr()

similarity_weights_matrix = (
    top20_similarity.dot(rating_presence)
).tocsr()

print("Weighted deviations:", weighted_deviations_matrix.shape)
print("Similarity weights:", similarity_weights_matrix.shape)

Weighted deviations: (6040, 3706)
Similarity weights: (6040, 3706)


In [20]:
user_id = 1

user_index = user_movie_matrix.index.get_loc(user_id)

numerator = (
    weighted_deviations_matrix
    .getrow(user_index)
    .toarray()
    .ravel()
)

denominator = (
    similarity_weights_matrix
    .getrow(user_index)
    .toarray()
    .ravel()
)

user_mean = user_means.loc[user_id]

# Only retain movies with actual collaborative evidence
valid_predictions = denominator > 0

predicted_ratings = np.full(
    len(user_movie_matrix.columns),
    np.nan,
    dtype=np.float32
)

predicted_ratings[valid_predictions] = (
    user_mean +
    numerator[valid_predictions] / denominator[valid_predictions]
)

predicted_ratings[valid_predictions] = np.clip(
    predicted_ratings[valid_predictions],
    1,
    5
)

collaborative_scores = pd.Series(
    predicted_ratings,
    index=user_movie_matrix.columns,
    name="PredictedRating"
)

# Remove movies without collaborative predictions
collaborative_scores = collaborative_scores.dropna()

# Exclude movies already rated by this user
collaborative_scores = collaborative_scores[
    ~collaborative_scores.index.isin(watched_movie_ids)
]

collaborative_recommendations = (
    collaborative_scores
    .sort_values(ascending=False)
    .reset_index()
)

collaborative_recommendations.columns = [
    "MovieID",
    "PredictedRating"
]

collaborative_recommendations = collaborative_recommendations.merge(
    movies.reset_index()[["MovieID", "Title", "Genres"]],
    on="MovieID",
    how="left"
)

collaborative_recommendations.head(10)

,MovieID,PredictedRating,Title,Genres
0,3916,5.0,Remember the Titans (2000),Drama
1,2,5.0,Jumanji (1995),Adventure|Children's|Fantasy
2,94,5.0,Beautiful Girls (1996),Drama
3,3654,5.0,"Guns of Navarone, The (1961)",Action|Drama|War
4,62,5.0,Mr. Holland's Opus (1995),Drama
5,3448,5.0,"Good Morning, Vietnam (1987)",Comedy|Drama|War
6,3478,5.0,"Bamba, La (1987)",Drama
7,3536,5.0,Keeping the Faith (2000),Comedy|Romance
8,3608,5.0,Pee-wee's Big Adventure (1985),Comedy
9,3476,5.0,Jacob's Ladder (1990),Horror|Mystery|Thriller


In [21]:
collaborative_recommendations["CollaborativeScore"] = (
    collaborative_recommendations["PredictedRating"] - 1
) / 4

collaborative_recommendations.head(10)

,MovieID,PredictedRating,Title,Genres,CollaborativeScore
0,3916,5.0,Remember the Titans (2000),Drama,1.0
1,2,5.0,Jumanji (1995),Adventure|Children's|Fantasy,1.0
2,94,5.0,Beautiful Girls (1996),Drama,1.0
3,3654,5.0,"Guns of Navarone, The (1961)",Action|Drama|War,1.0
4,62,5.0,Mr. Holland's Opus (1995),Drama,1.0
5,3448,5.0,"Good Morning, Vietnam (1987)",Comedy|Drama|War,1.0
6,3478,5.0,"Bamba, La (1987)",Drama,1.0
7,3536,5.0,Keeping the Faith (2000),Comedy|Romance,1.0
8,3608,5.0,Pee-wee's Big Adventure (1985),Comedy,1.0
9,3476,5.0,Jacob's Ladder (1990),Horror|Mystery|Thriller,1.0


In [22]:
hybrid_recommendations = content_recommendations.merge(
    collaborative_recommendations[
        ["MovieID", "CollaborativeScore", "PredictedRating"]
    ],
    on="MovieID",
    how="inner"
)

print("Movies with scores from both models:", len(hybrid_recommendations))

Movies with scores from both models: 746


In [23]:
alpha = 0.5

hybrid_recommendations["HybridScore"] = (
    alpha * hybrid_recommendations["ContentScore"] +
    (1 - alpha) * hybrid_recommendations["CollaborativeScore"]
)

hybrid_recommendations = hybrid_recommendations.sort_values(
    "HybridScore",
    ascending=False
).reset_index(drop=True)

hybrid_recommendations.head(10)

,MovieID,ContentScore,Title,Genres,CollaborativeScore,PredictedRating,HybridScore
0,1012,0.765737,Old Yeller (1957),Children's|Drama,1.000000,5.000000,0.882868
1,2102,0.763594,Steamboat Willie (1940),Animation|Children's|Comedy|Musical,0.993978,4.975914,0.878786
2,364,0.705378,"Lion King, The (1994)",Animation|Children's|Musical,0.994618,4.978472,0.849998
3,1032,0.705378,Alice in Wonderland (1951),Animation|Children's|Musical,0.993978,4.975914,0.849678
4,262,0.765737,"Little Princess, A (1995)",Children's|Drama,0.901336,4.605346,0.833537
5,1410,0.628297,"Evening Star, The (1996)",Comedy|Drama,1.000000,5.000000,0.814149
6,2252,0.628297,Hero (1992),Comedy|Drama,1.000000,5.000000,0.814149
7,1476,0.628297,Private Parts (1997),Comedy|Drama,1.000000,5.000000,0.814149
8,2961,0.628297,"Story of Us, The (1999)",Comedy|Drama,1.000000,5.000000,0.814149
9,2290,0.628297,Stardust Memories (1980),Comedy|Drama,1.000000,5.000000,0.814149


In [24]:
print("CONTENT-BASED RECOMMENDATIONS")
display(
    content_recommendations[
        ["Title", "ContentScore"]
    ].head(5)
)

print("\nCOLLABORATIVE RECOMMENDATIONS")
display(
    collaborative_recommendations[
        ["Title", "PredictedRating", "CollaborativeScore"]
    ].head(5)
)

print("\nHYBRID RECOMMENDATIONS")
display(
    hybrid_recommendations[
        ["Title", "ContentScore", "CollaborativeScore", "HybridScore"]
    ].head(10)
)

CONTENT-BASED RECOMMENDATIONS


,Title,ContentScore
0,Wide Awake (1998),0.801566
1,Babe (1995),0.801566
2,Pollyanna (1960),0.801566
3,Watership Down (1978),0.777478
4,"Little Princess, A (1995)",0.765737



COLLABORATIVE RECOMMENDATIONS


,Title,PredictedRating,CollaborativeScore
0,Remember the Titans (2000),5.0,1.0
1,Jumanji (1995),5.0,1.0
2,Beautiful Girls (1996),5.0,1.0
3,"Guns of Navarone, The (1961)",5.0,1.0
4,Mr. Holland's Opus (1995),5.0,1.0



HYBRID RECOMMENDATIONS


,Title,ContentScore,CollaborativeScore,HybridScore
0,Old Yeller (1957),0.765737,1.000000,0.882868
1,Steamboat Willie (1940),0.763594,0.993978,0.878786
2,"Lion King, The (1994)",0.705378,0.994618,0.849998
3,Alice in Wonderland (1951),0.705378,0.993978,0.849678
4,"Little Princess, A (1995)",0.765737,0.901336,0.833537
5,"Evening Star, The (1996)",0.628297,1.000000,0.814149
6,Hero (1992),0.628297,1.000000,0.814149
7,Private Parts (1997),0.628297,1.000000,0.814149
8,"Story of Us, The (1999)",0.628297,1.000000,0.814149
9,Stardust Memories (1980),0.628297,1.000000,0.814149


In [25]:
print("Predicted rating statistics:")
print(collaborative_scores.describe())

print("\nPredictions at minimum rating:")
print((collaborative_scores == 1).sum())

print("\nPredictions at maximum rating:")
print((collaborative_scores == 5).sum())

print("\nNumber of distinct predicted ratings:")
print(collaborative_scores.nunique())

Predicted rating statistics:
count    746.000000
mean       4.017572
std        0.868767
min        1.139899
25%        3.524334
50%        4.196112
75%        4.733167
max        5.000000
Name: PredictedRating, dtype: float64

Predictions at minimum rating:
0

Predictions at maximum rating:
114

Number of distinct predicted ratings:
275


In [26]:
hybrid_recommendations = content_recommendations.merge(
    collaborative_recommendations[
        ["MovieID", "CollaborativeScore", "PredictedRating"]
    ],
    on="MovieID",
    how="inner"
)

alpha = 0.5

hybrid_recommendations["HybridScore"] = (
    alpha * hybrid_recommendations["ContentScore"] +
    (1 - alpha) * hybrid_recommendations["CollaborativeScore"]
)

hybrid_recommendations = hybrid_recommendations.sort_values(
    "HybridScore",
    ascending=False
).reset_index(drop=True)

hybrid_recommendations[
    [
        "MovieID",
        "Title",
        "Genres",
        "ContentScore",
        "CollaborativeScore",
        "PredictedRating",
        "HybridScore"
    ]
].head(10)

,MovieID,Title,Genres,ContentScore,CollaborativeScore,PredictedRating,HybridScore
0,1012,Old Yeller (1957),Children's|Drama,0.765737,1.000000,5.000000,0.882868
1,2102,Steamboat Willie (1940),Animation|Children's|Comedy|Musical,0.763594,0.993978,4.975914,0.878786
2,364,"Lion King, The (1994)",Animation|Children's|Musical,0.705378,0.994618,4.978472,0.849998
3,1032,Alice in Wonderland (1951),Animation|Children's|Musical,0.705378,0.993978,4.975914,0.849678
4,262,"Little Princess, A (1995)",Children's|Drama,0.765737,0.901336,4.605346,0.833537
5,1410,"Evening Star, The (1996)",Comedy|Drama,0.628297,1.000000,5.000000,0.814149
6,2252,Hero (1992),Comedy|Drama,0.628297,1.000000,5.000000,0.814149
7,1476,Private Parts (1997),Comedy|Drama,0.628297,1.000000,5.000000,0.814149
8,2961,"Story of Us, The (1999)",Comedy|Drama,0.628297,1.000000,5.000000,0.814149
9,2290,Stardust Memories (1980),Comedy|Drama,0.628297,1.000000,5.000000,0.814149


In [27]:
print("Hybrid score statistics:")
print(hybrid_recommendations["HybridScore"].describe())

print("\nMissing hybrid scores:")
print(hybrid_recommendations["HybridScore"].isna().sum())

print("\nDuplicate movie IDs:")
print(hybrid_recommendations["MovieID"].duplicated().sum())

print("\nAlready watched movies in recommendations:")
print(
    hybrid_recommendations["MovieID"]
    .isin(watched_movie_ids)
    .sum()
)

Hybrid score statistics:
count    746.000000
mean       0.555936
std        0.148816
min        0.070400
25%        0.457931
50%        0.570351
75%        0.657713
max        0.882868
Name: HybridScore, dtype: float64

Missing hybrid scores:
0

Duplicate movie IDs:
0

Already watched movies in recommendations:
0


In [28]:
def recommend_hybrid(user_id, top_n=10, alpha=0.5):

    if not 0 <= alpha <= 1:
        raise ValueError("Alpha must be between 0 and 1.")

    if user_id not in user_movie_matrix.index:
        raise ValueError("User ID not found.")

    # Get this user's rating history
    user_ratings = ratings[
        ratings["UserID"] == user_id
    ]

    watched_movie_ids = set(user_ratings["MovieID"])

    # Identify movies the user liked
    liked_movie_ids = user_ratings.loc[
        user_ratings["Rating"] >= 4,
        "MovieID"
    ]

    liked_movie_ids = genre_features.index.intersection(
        liked_movie_ids
    )

    if len(liked_movie_ids) == 0:
        raise ValueError("User has no liked movies for content profiling.")

    # Build content profile
    user_profile = genre_features.loc[
        liked_movie_ids
    ].mean(axis=0)

    # Calculate content similarities
    content_values = cosine_similarity(
        user_profile.values.reshape(1, -1),
        genre_features
    )[0]

    content_scores = pd.Series(
        content_values,
        index=genre_features.index
    )

    # Get collaborative predictions
    user_index = user_movie_matrix.index.get_loc(user_id)

    numerator = (
        weighted_deviations_matrix
        .getrow(user_index)
        .toarray()
        .ravel()
    )

    denominator = (
        similarity_weights_matrix
        .getrow(user_index)
        .toarray()
        .ravel()
    )

    valid_predictions = denominator > 0

    predicted_ratings = np.full(
        len(user_movie_matrix.columns),
        np.nan,
        dtype=np.float32
    )

    predicted_ratings[valid_predictions] = (
        user_means.loc[user_id] +
        numerator[valid_predictions] / denominator[valid_predictions]
    )

    predicted_ratings[valid_predictions] = np.clip(
        predicted_ratings[valid_predictions],
        1,
        5
    )

    collaborative_scores = pd.Series(
        predicted_ratings,
        index=user_movie_matrix.columns
    ).dropna()

    # Normalize collaborative scores
    collaborative_scores = (
        collaborative_scores - 1
    ) / 4

    # Combine both scoring methods
    hybrid_scores = pd.concat(
        [
            content_scores.rename("ContentScore"),
            collaborative_scores.rename("CollaborativeScore")
        ],
        axis=1,
        join="inner"
    )

    # Remove already watched movies
    hybrid_scores = hybrid_scores.drop(
        index=list(watched_movie_ids),
        errors="ignore"
    )

    hybrid_scores["HybridScore"] = (
        alpha * hybrid_scores["ContentScore"] +
        (1 - alpha) * hybrid_scores["CollaborativeScore"]
    )

    # Rank recommendations
    recommendations = hybrid_scores.sort_values(
        "HybridScore",
        ascending=False
    ).head(top_n)

    recommendations = recommendations.reset_index()
    recommendations = recommendations.rename(
        columns={"index": "MovieID"}
    )

    # Attach movie details
    recommendations = recommendations.merge(
        movies.reset_index()[["MovieID", "Title", "Genres"]],
        on="MovieID",
        how="left"
    )

    return recommendations[
        [
            "MovieID",
            "Title",
            "Genres",
            "ContentScore",
            "CollaborativeScore",
            "HybridScore"
        ]
    ]

In [29]:
recommend_hybrid(user_id=1, top_n=10)

,MovieID,Title,Genres,ContentScore,CollaborativeScore,HybridScore
0,1012,Old Yeller (1957),Children's|Drama,0.765737,1.000000,0.882868
1,2102,Steamboat Willie (1940),Animation|Children's|Comedy|Musical,0.763594,0.993978,0.878786
2,364,"Lion King, The (1994)",Animation|Children's|Musical,0.705378,0.994618,0.849998
3,1032,Alice in Wonderland (1951),Animation|Children's|Musical,0.705378,0.993978,0.849678
4,262,"Little Princess, A (1995)",Children's|Drama,0.765737,0.901336,0.833537
5,3255,"League of Their Own, A (1992)",Comedy|Drama,0.628297,1.000000,0.814149
6,2289,"Player, The (1992)",Comedy|Drama,0.628297,1.000000,0.814149
7,2961,"Story of Us, The (1999)",Comedy|Drama,0.628297,1.000000,0.814149
8,2290,Stardust Memories (1980),Comedy|Drama,0.628297,1.000000,0.814149
9,2324,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama,0.628297,1.000000,0.814149


In [30]:
recommend_hybrid(user_id=10, top_n=10)

,MovieID,Title,Genres,ContentScore,CollaborativeScore,HybridScore
0,892,Twelfth Night (1996),Comedy|Drama|Romance,0.754143,1.000000,0.877071
1,2171,"Next Stop, Wonderland (1998)",Comedy|Drama|Romance,0.754143,1.000000,0.877071
2,1172,Cinema Paradiso (1988),Comedy|Drama|Romance,0.754143,1.000000,0.877071
3,1542,Brassed Off (1996),Comedy|Drama|Romance,0.754143,0.989548,0.871845
4,3504,Network (1976),Comedy|Drama,0.736941,1.000000,0.868470
5,3556,"Virgin Suicides, The (1999)",Comedy|Drama,0.736941,1.000000,0.868470
6,2352,"Big Chill, The (1983)",Comedy|Drama,0.736941,1.000000,0.868470
7,2506,"Other Sister, The (1999)",Comedy|Drama|Romance,0.754143,0.981362,0.867752
8,1642,Indian Summer (a.k.a. Alive & Kicking) (1996),Comedy|Drama,0.736941,0.989548,0.863244
9,2596,SLC Punk! (1998),Comedy|Drama,0.736941,0.981362,0.859151


In [31]:
recommend_hybrid(user_id=100, top_n=10)

,MovieID,Title,Genres,ContentScore,CollaborativeScore,HybridScore
0,1287,Ben-Hur (1959),Action|Adventure|Drama,0.900997,0.826929,0.863963
1,522,Romper Stomper (1992),Action|Drama,0.788208,0.782752,0.785480
2,2370,"Emerald Forest, The (1985)",Action|Adventure|Drama,0.900997,0.659357,0.780177
3,1277,Cyrano de Bergerac (1990),Action|Drama|Romance,0.746541,0.754912,0.750726
4,3360,Hoosiers (1986),Drama,0.535054,0.903300,0.719177
5,848,"Spitfire Grill, The (1996)",Drama,0.535054,0.903300,0.719177
6,3071,Stand and Deliver (1987),Drama,0.535054,0.903300,0.719177
7,3578,Gladiator (2000),Action|Drama,0.788208,0.639679,0.713944
8,1233,"Boat, The (Das Boot) (1981)",Action|Drama|War,0.772283,0.655585,0.713934
9,3153,"7th Voyage of Sinbad, The (1958)",Action|Adventure|Fantasy,0.617827,0.793421,0.705624


In [32]:
# Equal contribution
recommend_hybrid(user_id=1, alpha=0.5)

# More influence from content-based filtering
recommend_hybrid(user_id=1, alpha=0.7)

# More influence from collaborative filtering
recommend_hybrid(user_id=1, alpha=0.3)

,MovieID,Title,Genres,ContentScore,CollaborativeScore,HybridScore
0,1012,Old Yeller (1957),Children's|Drama,0.765737,1.000000,0.929721
1,2102,Steamboat Willie (1940),Animation|Children's|Comedy|Musical,0.763594,0.993978,0.924863
2,364,"Lion King, The (1994)",Animation|Children's|Musical,0.705378,0.994618,0.907846
3,1032,Alice in Wonderland (1951),Animation|Children's|Musical,0.705378,0.993978,0.907398
4,3255,"League of Their Own, A (1992)",Comedy|Drama,0.628297,1.000000,0.888489
5,2352,"Big Chill, The (1983)",Comedy|Drama,0.628297,1.000000,0.888489
6,2961,"Story of Us, The (1999)",Comedy|Drama,0.628297,1.000000,0.888489
7,2324,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama,0.628297,1.000000,0.888489
8,2290,Stardust Memories (1980),Comedy|Drama,0.628297,1.000000,0.888489
9,2289,"Player, The (1992)",Comedy|Drama,0.628297,1.000000,0.888489
